# Demo: View feedforward point cloud (NPZ)

Load a colored point cloud from `predictions.npz` and inspect it interactively with Plotly.

**Usage:**
1. Run `./run_lingbot.sh` (or any feedforward engine) to produce `predictions.npz`.
2. Set `NPZ_PATH` below to that file.
3. Run all cells.

**Example output layout:**
```text
feedforward_output/lingbot_test6/predictions.npz
```

Large clouds are downsampled before plotting so the notebook stays responsive.


In [ ]:
from pathlib import Path

# Default: NPZ from a recent feedforward run (change as needed)
NPZ_PATH = Path("../../feedforward_output/lingbot_map_test/predictions.npz").resolve()

# Plotly gets slow above ~200k points in the browser
MAX_POINTS = 200_000

assert NPZ_PATH.exists(), f"NPZ not found: {NPZ_PATH}"
NPZ_PATH


In [ ]:
import sys

import numpy as np

REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from vibephysics.feedforward.common import collect_colored_point_cloud, resolve_confidence_threshold
from vibephysics.feedforward.schema import load_prediction


def downsample(
    positions: np.ndarray,
    colors: np.ndarray,
    max_points: int,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray]:
    if len(positions) <= max_points:
        return positions, colors
    idx = np.random.default_rng(seed).choice(len(positions), size=max_points, replace=False)
    return positions[idx], colors[idx]


prediction = load_prediction(NPZ_PATH)
min_confidence = resolve_confidence_threshold(prediction, min_confidence=0.5)
positions, colors, _, _ = collect_colored_point_cloud(
    prediction,
    min_confidence=min_confidence,
    to_blender=True,
)
positions, colors = downsample(positions, colors, MAX_POINTS)

print(f"Points in view: {len(positions):,}")
print(f"Bounds min: {positions.min(axis=0)}")
print(f"Bounds max: {positions.max(axis=0)}")


In [ ]:
try:
    import plotly.graph_objects as go
except ImportError:
    %pip install plotly -q
    import plotly.graph_objects as go

# Per-point RGB for Plotly (downsampled set is small enough for a list comp)
color_strings = [f"rgb({int(r)},{int(g)},{int(b)})" for r, g, b in colors]

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=positions[:, 0],
            y=positions[:, 1],
            z=positions[:, 2],
            mode="markers",
            marker=dict(size=1.5, color=color_strings, opacity=0.95),
        )
    ]
)
fig.update_layout(
    title=f"Feedforward point cloud: {NPZ_PATH.name}",
    scene=dict(aspectmode="data", xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    margin=dict(l=0, r=0, t=40, b=0),
    height=720,
)
fig.show()
